In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium.common.exceptions import TimeoutException
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import time
general_df = pd.read_excel(r'C:\Users\Coco\OneDrive - Erasmus University Rotterdam\Coding\Betting hltv\player_stats\general_df_test.xlsx')

In [ ]:
def open_stats_links(general_df, n):
    stats_list = []
    for index in range(0,5):                       #uncomment when testing and remove the other loop
    # Iterate through all the rows in general_df
    #for index, row in general_df.iterrows():                           
        row=general_df.loc[index]
        try:
                # Set up the WebDriver
                driver = webdriver.Firefox()
                # Get the match link of the current row
                match_link = row['Match Link']

                # Navigate to the match page
                driver.get(match_link)

                # Locate all the STATS links
                elements = WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div.results-center-stats > a.results-stats')))

                # Keep track of all the open tabs
                tabs = [driver.current_window_handle]

                # Keep track of the current map number
                map_num = 1

                # Iterate through the STATS links and click on them
                for element in elements:
                    if element.text == "STATS":
                        stats_link = element.get_attribute("href")
                        print(f"Opening STATS link: {stats_link}")

                        # Open the STATS link in a new tab
                        driver.execute_script("window.open('');")

                        # Switch to the new tab
                        driver.switch_to.window(driver.window_handles[-1])

                        # Navigate to the correct STATS link in the new tab
                        driver.get(stats_link)

                        # Scrape the player statistics for the current tab
                        player_stats = scrape_player_stats(driver, map_num)

                        # Add the player statistics to the stats_list
                        stats_list.extend(player_stats)

                        # Print the player statistics for debugging purposes
                        print(f'Player statistics for map {map_num}:')
                        for stat in player_stats:
                            print(stat)
                            print('-' * 50)

                        # Add the new tab to the list of open tabs
                        tabs.append(driver.current_window_handle)

                        # Increment the map number
                        map_num += 1

                        # Add a delay to ensure that the page loads completely
                        time.sleep(2)

                        # Close the current tab
                        driver.close()

                        # Switch back to the original tab
                        driver.switch_to.window(tabs[0])

                # Close the original tab
                driver.close()

                # Remove the closed tab from the list of open tabs
                tabs.remove(tabs[0])

                # Close the WebDriver
                driver.quit()
        except TimeoutException:
            print(f'TimeoutException occurred for {match_link} skipping to the next match.')
            driver.quit()
            pass
    return stats_list

def scrape_player_stats(driver, map_num):
    # Find all the tables with the player statistics
    tables = driver.find_elements(By.CSS_SELECTOR, 'div.stats-content > table.stats-table.totalstats')
    player_stats = []

    for table in tables:
        # Find all the rows in the table
        rows = table.find_elements(By.CSS_SELECTOR, 'tbody > tr')

        # Iterate through each row and extract the player statistics
        
        for row in rows:
            # Find all the columns in the row
            columns = row.find_elements(By.TAG_NAME, 'td')

            # Extract the player name
            player_name = columns[0].find_element(By.CSS_SELECTOR, 'a').get_attribute('textContent')
            print(f'Found player name: {player_name}')

            # Extract the kills
            kills = columns[1].get_attribute('textContent')
            print(f'Found kills: {kills}')

            # Extract the assists
            assists = columns[2].get_attribute('textContent')
            print(f'Found assists: {assists}')

            # Extract the deaths
            deaths = columns[3].get_attribute('textContent')
            print(f'Found deaths: {deaths}')

            # Extract the K/D ratio
            kdratio = columns[4].get_attribute('textContent')
            print(f'Found K/D ratio: {kdratio}')

            # Extract the K/D difference
            kddiff = columns[5].get_attribute('data-sort')
            print(f'Found K/D difference: {kddiff}')

            # Extract the average damage per round
            adr = columns[6].get_attribute('textContent')
            print(f'Found average damage per round: {adr}')

            # Extract the first kill difference
            fkdiff = columns[7].get_attribute('textContent')
            print(f'Found first kill difference: {fkdiff}')

            # Extract the rating
            rating = columns[8].get_attribute('textContent')
            print(f'Found rating: {rating}')

            # Print a separator for clarity
            print('-' * 50)

            # Create a dictionary with the player statistics and map number
            player_dict = {
                'Player': f'{player_name}',
                'Map': map_num,
                'Kills': kills,
                'Assists': assists,
                'Deaths': deaths,
                'K/D Ratio': kdratio,
                'K/D Difference': kddiff,
                'Average Damage per Round': adr,
                'First Kill Difference': fkdiff,
                'Rating': rating
            }

            # Add the dictionary to the list of player statistics
            player_stats.append(player_dict)

    return player_stats


In [ ]:

# Initialize an empty DataFrame
player_stats_df = pd.DataFrame()
# Iterate through the list of dictionaries
for i, player_stat in enumerate(stats_list):
    # Create a new DataFrame from the current dictionary
    temp_df = pd.DataFrame([player_stat])

    # Merge the new DataFrame with the existing DataFrame
    player_stats_df = pd.concat([player_stats_df, temp_df], ignore_index=True)

# Print the DataFrame
player_stats_df

def clean_kills(kills_str):
    if "(" in kills_str and ")" in kills_str:
        kills, hs_kills = kills_str.split(" (")
        hs_kills = hs_kills.replace(")", "")
        return int(kills), int(hs_kills)
    else:
        return int(kills_str), None

# Assuming 'player_stats_df' is your DataFrame
kills, hs_kills = zip(*player_stats_df["Kills"].apply(clean_kills))
player_stats_df["Kills"] = kills
player_stats_df["hs_kills"] = hs_kills

def clean_assists(assists_str):
    if "(" in assists_str and ")" in assists_str:
        assists, flash_assists = assists_str.split(" (")
        flash_assists = flash_assists.replace(")", "")
        return int(assists), int(flash_assists)
    else:
        return int(assists_str), None

# Assuming 'player_stats_df' is your DataFrame
assists, flash_assists = zip(*player_stats_df["Assists"].apply(clean_assists))
player_stats_df["Assists"] = assists
player_stats_df["flash_assists"] = flash_assists

# Print the updated DataFrame
player_stats_df
# Convert the columns to integers
int_columns = ['K/D Difference', 'First Kill Difference', 'Deaths', 'Assists', 'Kills', 'Map']
player_stats_df[int_columns] = player_stats_df[int_columns].astype(int)

# Convert the columns to floats
float_columns = ['Average Damage per Round', 'Rating']
player_stats_df[float_columns] = player_stats_df[float_columns].astype(float)
# Remove the percentage sign and convert the column to floats
player_stats_df['K/D Ratio'] = player_stats_df['K/D Ratio'].str.replace('%', '').astype(float)
player_stats_df=player_stats_df.rename(columns={"K/D Ratio": "K/D Ratio (%)"})
player_stats_df

# Initialize an empty dictionary to store the player statistics for each map
player_stats_dict = {}

# Iterate over the rows of the player_stats_df DataFrame
for index, row in player_stats_df.iterrows():
    # Get the map number for the current row
    map_num = f"Map {row['Map']}"
    
    # Check if the map number is already a key in the player_stats_dict
    if map_num not in player_stats_dict:
        # If the map number is not a key, initialize an empty dictionary for that map
        player_stats_dict[map_num] = {}
    
    # Create a nested dictionary for the player statistics
    player_stats = {
        'Kills': row['Kills'],
        'Assists': row['Assists'],
        'Deaths': row['Deaths'],
        'K/D Ratio (%)': row['K/D Ratio (%)'],
        'K/D Difference': row['K/D Difference'],
        'Average Damage per Round': row['Average Damage per Round'],
        'First Kill Difference': row['First Kill Difference'],
        'Rating': row['Rating']
    }
    
    # Add the nested dictionary to the player_stats_dict for the corresponding map
    player_stats_dict[map_num][row['Player']] = player_stats
    
player_stats_dict
player_stats_df

In [ ]:
# Assuming player_stats_dict is your dictionary containing player statistics

# Select the "Map 1" dictionary
map_1_dict = player_stats_dict.get('Map 1')

# Select the first dictionary value from the "Map 1" dictionary
first_player_stats = next(iter(map_1_dict.values()))

print(first_player_stats)


In [ ]:
import itertools

# Get an iterator of the dictionary items (key-value pairs)
map_1_items = iter(map_1_dict.items())

# Use islice to get the first 10 items
first_10_players = list(itertools.islice(map_1_items, 10))

print(first_10_players)


In [ ]:
player_stats_dict["Map 1"]

In [ ]:
import itertools
import math

# Create columns in general_df for player stats maps
num_maps = len(player_stats_dict)
for i in range(1, num_maps + 1):
    column_name = f"Player Stats Map {i}"
    general_df[column_name] = None

# Get the number of rows in general_df
num_rows = len(general_df)

# Iterate through the maps in player_stats_dict
for map_num, map_dict in player_stats_dict.items():
    # Extract the map number from the key (e.g., "Map 1" -> 1)
    map_number = int(map_num.split()[-1])

    # Calculate the number of chunks (lists of 10 players) needed for the current map
    num_chunks = math.ceil(len(map_dict) / 10)

    # Ensure there are enough rows in general_df for the current map's data
    if num_rows < num_chunks:
        general_df = general_df.append([{}] * (num_chunks - num_rows), ignore_index=True)
        num_rows = len(general_df)

    # Create chunks (lists of 10 players) for the current map
    map_chunks = [list(map_dict.items())[i:i+10] for i in range(0, len(map_dict), 10)]

    # Add each chunk to the appropriate column in general_df
    for i, chunk in enumerate(map_chunks):
        column_name = f"Player Stats Map {map_number}"
        general_df.at[i, column_name] = chunk


In [ ]:
general_df.head()

In [ ]:
for map_num in player_stats_dict[f"Map {map_num}"]:
    

In [ ]:
#general_df = pd.read_excel(r'C:\Users\Coco\OneDrive - Erasmus University Rotterdam\Coding\Betting hltv\player_stats\general_df_test.xlsx')

# Iterate through the maps in player_stats_dict
for map_num, map_stats in player_stats_dict.items():
    # Add a new column to the general_df DataFrame
    general_df[f"Player Stats {map_num}"] = None

    # Iterate through the rows in general_df
    for index, row in general_df.iterrows():
        # Get the player statistics for the current map
        player_stats = map_stats

        # Add the player statistics dictionary to the general_df DataFrame
        general_df.at[index, f"Player Stats {map_num}"] = player_stats

general_df.head()